# Projeto 1 - T320 (1S2026)

### Instruções:

1. Quando você terminar os exercícios do projeto, vá até o menu do Jupyter ou Colab e selecione a opção para fazer download do notebook.
    * Os notebooks tem extensão .ipynb.
    * Este será o arquivo que você entregará após a finalização dos exercícios.
    * Para baixá-lo, no Colab, vá até a opção **Arquivo** -> **Fazer download** -> **Baixar o .ipynb**.
2. Após o download do notebook, vá até a aba de tarefas do MS Teams, localize a tarefa referente a este projeto e faça o upload do seu notebook. Veja que há uma opção de anexar arquivos à tarefa.
3. Atente-se ao prazo de entrega definido na tarefa do MS Teams. Entregas fora do prazo não serão aceitas.
4. **O projeto pode ser resolvido em grupos de no MÁXIMO 3 alunos**.
5. Todas as questões têm o mesmo peso.
6. Não se esqueça de colocar seu(s) nome(s) e número(s) de matrícula no campo abaixo. Substitua os nomes que já estão no campo abaixo.
7. Você pode consultar todo o material de aula.
8. A interpretação faz parte do projeto. Leia o enunciado de cada questão atentamente!
9. Boa sorte!

---



**Nomes e matrículas**:

1.

2.

# 1) **PROJETO SOBRE CLASSIFICAÇÃO MULTICLASSES APLICADA À VISÃO COMPUTACIONAL.**

**CONTEXTO: Você foi contratado por uma empresa que atua no mercado de sistemas de visão computacional. O seu objetivo é desenvolver um sistema capaz de identificar automaticamente o tipo de veículo apenas a partir de sua silhueta 2D. O sistema deve receber 18 descritores geométricos e estatísticos extraídos da imagem da silhueta (compactness, circularity, elongatedness, momentos estatísticos, etc.) e classificar o veículo em uma das 04 categorias descritas abaixo:**

* OPEL;

* SAAB;

* BUS;

* VAN.

Para isto, neste projeto, nós vamos considerar o problema de classificar exemplos de veículos tendo como base os atributos extraídos de imagens de suas silhuetas.

Inicialmente, o nosso conjunto de dados já estará dividido em duas partes: Uma para treinamento (recomendado: 70% a 80%) e outra reservada para validação/teste (restante).

Dois métodos de classificação serão explorados neste exercício:

* Regressão Logística com abordagem Um-Contra-o-Resto (abordagem One-vs-Rest - OvR): Treinar um classificador binário para cada classe contra todas as demais;

* Regressão Logística Multimodal (Softmax): Utilizar a função Softmax para modelar diretamente as probabilidades das quatro classes em um único modelo para lidar com este cenário de classificação multi-classe.

A abordagem do problema de classificação multiclasse utilizando os dados do repositório Statlog (Vehicle Silhouettes) encontra-se disponível no UCI Machine Learning Repository, que pode ser acessada através dos seguintes links:

https://archive.ics.uci.edu/ml/datasets/Statlog+(Vehicle+Silhouettes)

https://www.kaggle.com/datasets/pritech/vehicle-silhouettes


A base de dados possui 846 exemplos, onde cada exemplo (veículo) é descrito por 18 atributos numéricos (mostrados na tabela abaixo), além das 04 classes já disponíveis acima.

|             Atributo             |                       Descrição                       |
|:--------------------------------:|:-----------------------------------------------------:|
|           COMPACTNESS            |                (average perim)**2/area                |
|           CIRCULARITY            |                (average radius)**2/area               |
|       DISTANCE CIRCULARITY       |           area/(av.distance from border)**2           |
|           RADIUS RATIO           |              (max.rad-min.rad)/av.radius              |
|       PR.AXIS ASPECT RATIO       |               (minor axis)/(major axis)               |
|     MAX.LENGTH ASPECT RATIO      |         (length perp. max length)/(max length)        |
|          SCATTER RATIO           | (inertia about minor axis)/(inertia about major axis) |
|          ELONGATEDNESS           |                 area/(shrink width)**2                |
|      PR.AXIS RECTANGULARITY      |          area/(pr.axis length*pr.axis width)          |
|    MAX.LENGTH RECTANGULARITY     |         area/(max.length*length perp. to this)        |
| SCALED VARIANCE ALONG MAJOR AXIS |        (2nd order moment about minor axis)/area       |
| SCALED VARIANCE ALONG MINOR AXIS |        (2nd order moment about major axis)/area       |
|    SCALED RADIUS OF GYRATION     |                   (mavar+mivar)/area                  |
|     SKEWNESS ABOUT MAJOR AXIS    |    (3rd order moment about major axis)/sigma_min**3   |
|     SKEWNESS ABOUT MINOR AXIS    |    (3rd order moment about minor axis)/sigma_maj**3   |
|     KURTOSIS ABOUT MINOR AXIS    |    (4th order moment about major axis)/sigma_min**4   |
|     KURTOSIS ABOUT MAJOR AXIS    |    (4th order moment about minor axis)/sigma_maj**4   |
|          HOLLOWS RATIO           |      (area of hollows)/(area of bounding polygon)     |

A figura abaixo mostra exemplos das quatro classes possíveis.

<img src="https://github.com/zz4fap/t320_aprendizado_de_maquina/blob/main/figures/carros_dataset.PNG?raw=1" width="500px">

1. Execute a célula de código abaixo para gerar os conjuntos de treinamento e validação.

**DICAS:**
+ Perceba que os dados já são separados em conjuntos de trainamento e validação. Com 80% para treinamento e 20% para validação.

In [ ]:
# Import all the necessary libraries.
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LogisticRegressionCV
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, roc_curve, auc, f1_score
from sklearn.metrics import classification_report, precision_score, recall_score

# Reset PN sequence generator.
seed = 42
np.random.seed(seed)

# Define the number of classes.
numberOfClasses = 4

# Load the dataset.
df = pd.read_csv('https://www.dropbox.com/s/pe0v72fhd14ivao/dataset_vehicle.csv?dl=1')

# Convert the labels from strings into integer numbers.
c = {"van":0, "saab":1, "bus":2, "opel":3}
df.Class = [c[item] for item in df.Class]

# Convert dataset into numpy array.
data = df.to_numpy()

# Create attribute matrix and label vector.
X = data[:,0:18]
y = data[:,18]

# Split the data into training and validation datasets.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)

# Show the first five examples of the dataset.
df.head(20)

2. Execute a célula de código abaixo para exibir as dimensões dos dados e a distribuição de frequência de cada classe, permitindo verificar o balanceamento entre as 04 categorias.

In [ ]:
# Create attribute matrix and label vector (já feito no seu código)
X = data[:, 0:18]
y = data[:, 18]

# Ver dimensões
print("Shape de X:", X.shape)
print("Shape de y:", y.shape)

# Distribuição das classes
print("\nDistribuição das classes:")
print(pd.Series(y).value_counts().sort_index())

sns.countplot(x=df['Class'])
plt.title("Distribuição das Classes")
plt.show()

3. Neste item, você irá classificar a base de dados com o Regressor Logístico utilizando a abordagem **Um-Contra-o-Resto (One-versus-Rest - OvR)**.

Instancie um objeto da classe `OneVsRestClassifier` combinado com `LogisticRegression`, treine o modelo com o conjunto de treinamento, realize as predições no conjunto de validação (teste) e calcule/imprima a acurácia obtida.

**DICAS:**

+ Para implementar a estratégia One-versus-Rest (OvR), utilize a classe `OneVsRestClassifier` do módulo `sklearn.multiclass`, passando um objeto de `LogisticRegression` como estimador base.

+ Dentro do `LogisticRegression` implemente e configure os seguintes parâmetros:

  * `max_iter=10000`: Número máximo de iterações para evitar problemas de convergência;

  * `random_state=seed`: Define a semente aleatória para garantir reprodutibilidade dos resultados;

  * `n_jobs=-1`: Cada classificador binário será treinado utilizando apenas um núcleo de processamento.

+ O código para instanciar o classificador multiclasses usando a abordagem OvR com regressores logísticos (classificadores binários) é dado abaixo.

```python
model = OneVsRestClassifier(LogisticRegression(
        max_iter=10000,
        random_state=seed,
        n_jobs=-1
    )
)
```


In [ ]:
# Digite aqui o código do exercício.

4. Plote a matriz de confusão para o conjunto de validação (teste).

In [ ]:
# Digite aqui o código do exercício.

5. Analise a matriz de confusão e a figura com exemplos das quatro classes. O que você consegue concluir a respeito das **confusões** feitas pelo classificador?

**DICAS:**
+ Foque em classes similares e vejas as confusões feitas entre essas classes.

**Resposta:**

<span style="color:blue">Escreva abaixo a resposta do exercício.</span>

6. Use a função `classification_report` para imprimir as métricas deste classificador para o conjunto de validação (teste).

In [ ]:
# Digite aqui o código do exercício.

7. Neste item, você irá classificar a base de dados utilizando **Regressão Logística Multinomial (Regressor Softmax)**.

Utilize a função `make_pipeline` para criar um pipeline contendo `LogisticRegression`, treine o modelo com o conjunto de treinamento, realize as predições no conjunto de validação (teste), calculando e imprimindo a sua respectiva acurácia.

**DICAS:**

+ A classe `LogisticRegression` detecta automaticamente que o número de classes é maior do que 2 e usa o regressor softmax.
+ Dentro do `LogisticRegression`, configure os seguintes parâmetros:

  * `max_iter=10000`: Número máximo de iterações para evitar problemas de convergência;

  * `random_state=seed`: Define a semente aleatória para garantir reprodutibilidade dos resultados entre execuções;

  * `n_jobs=-1`: Utiliza todos os núcleos do processador disponíveis para acelerar o treinamento.


In [ ]:
# Digite aqui o código do exercício.

8. Plote a matriz de confusão para o conjunto de validação.

In [ ]:
# Digite aqui o código do exercício.

9. Use a função `classification_report` para imprimir as métricas deste classificador para o conjunto de validação (teste).

In [ ]:
# Digite aqui o código do exercício.

10. Qual dos dois classificadores (**OvR versus Softmax**) apresenta o melhor desempenho? Avalie os classificadores de forma global e também olhando para cada classe.

**DICAS:**

* Justifique sua resposta usando a métricas `Acurácia`, `Precisão`, `Recall` e `F1-Score`.



**Resposta:**

<span style="color:blue">Escreva abaixo a resposta do exercício.</span>

11. Se nós analisarmos as classes, percebe-se que as classes SAAB e OPEL poderiam ser agrupadas em um única classe chamada de CAR, pois ambos são carros e, em uma aplicação genérica, talvez não faça sentido classificar estes dois tipos de carros de forma independente. Assim, neste item, agrupamos as 02 classes em um única classe que será chamada de CAR.

Execute abaixo a célula de código para fazer o agrupamento das 02 classes.

Após a execução, a classe VAN será representada pelo número inteiro 0, a classe CAR (SAAB+OPEL) representada pelo número 1 e a classe BUS representada pelo número 2.

In [ ]:
# Change positions where the label is equal to 3 to 1, the number that will define the class CAR
y_train = np.where(y_train==3, 1, y_train)

# Change positions where the label is equal to 3 to 1, the number that will define the class CAR
y_test = np.where(y_test==3, 1, y_test)

print("=== Agrupamento de classes realizado com sucesso ===")
print(f"Classes únicas em y_train após agrupamento: {np.unique(y_train)}")
print(f"Classes únicas em y_test após agrupamento:  {np.unique(y_test)}\n")

print("Distribuição de classes - y_train:")
unique, counts = np.unique(y_train, return_counts=True)
for cls, count in zip(unique, counts):
    nome = "VAN" if cls == 0 else "CAR" if cls == 1 else "BUS"
    print(f"   Classe {cls} ({nome}): {count} amostras")

print("\nDistribuição de classes - y_test:")
unique, counts = np.unique(y_test, return_counts=True)
for cls, count in zip(unique, counts):
    nome = "VAN" if cls == 0 else "CAR" if cls == 1 else "BUS"
    print(f"   Classe {cls} ({nome}): {count} amostras")

# Verificação final de integridade
assert set(np.unique(y_train)) == {0, 1, 2}, "Erro: y_train deve conter apenas classes 0, 1 e 2"
assert set(np.unique(y_test))  == {0, 1, 2}, "Erro: y_test deve conter apenas classes 0, 1 e 2"

print("\n Agrupamento concluído corretamente:")
print("   VAN = 0 | CAR (SAAB + OPEL) = 1 | BUS = 2")

12. Refaça a classificação com a nova base de dados utilizando o classificador que foi selecionado como o melhor.

In [ ]:
# Digite aqui o código do exercício.

13. Plote a nova matriz de confusão para o conjunto de validação (teste).

**DICAS:**
+ Lembre-se que agora só temos apenas 03 classes: VAN (0), CAR (1) e BUS (2).

In [ ]:
# Digite aqui o código do exercício.

14. Imprima as métricas calculadas pela função `classification_report` das 03 classes para o conjunto de validação (teste).

In [ ]:
# Digite aqui o código do exercício.

15. Compare os novos resultados (acurácia, precisão, recall, f1-score e matriz de confusão) obtidos com uma base de dados com 3 classes com os mesmos resultados obtidos quando tínhamos 4 classes. O que pode ser concluído? (**Justifique sua resposta**)

**Resposta:**

<span style="color:blue">Escreva abaixo a resposta do exercício.</span>

# 2) **PROJETO SOBRE CLASSIFICAÇÃO APLICADA ÀS TELECOMUNICAÇÕES.**

**CONTEXTO: Você é um engenheiro de dados e foi contratado para desenvolver um sistema inteligente de demodulação de sinais modulados em 8-PSK utilizando classificadores de Machine Learning. Neste projeto, você irá gerar a constelação 8-PSK, simular o canal AWGN e treinar um classificador para recuperar os símbolos transmitidos a partir das componentes I e Q.**




1. Execute a célula abaixo e analise o resultado. A figura mostra símbolos ruidosos da modulação 8-PSK, onde cada um dos 08 possíveis símbolos é considerado como uma classe diferente.

**DICAS:**

+ Note que na célula de código abaixo, o conjunto de dados já está dividido em conjuntos de treinamento e validação.
+ Veja as referências abaixo.


#### **Referências:**

[1] '8-PSK', https://www.researchgate.net/publication/3158035_8-PSK_trellis_codes_for_a_Rayleigh_channel/link/0a85e5346717b0d657000000/download?_tp=eyJjb250ZXh0Ijp7ImZpcnN0UGFnZSI6InB1YmxpY2F0aW9uIiwicGFnZSI6InB1YmxpY2F0aW9uIn19

[2] 'Phase-shift_keying', https://en.wikipedia.org/wiki/Phase-shift_keying

In [ ]:
# Import all necessary modules.
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report
import seaborn as sns
from scipy.special import erfc
import time
from scipy.spatial import Voronoi, voronoi_plot_2d

# Reset PN sequence generator.
seed = 42
np.random.seed(seed)

# Função de modulação 8-PSK.
def modulate8PSK(bits):
    """
    Modula bits (0 a 7) em símbolos 8-PSK com energia normalizada para 1.
    """
    # Ângulos para 8-PSK: 0°, 45°, 90°, 135°, 180°, 225°, 270°, 315°
    angles = np.array([0, 45, 90, 135, 180, 225, 270, 315]) * np.pi / 180.0

    # Símbolos complexos (antes da normalização)
    ip = np.exp(1j * angles[bits])

    # Normalização de energia para 1 (dividir por sqrt(2) não é necessário aqui)
    symbol = ip / np.sqrt(1.0)   # |s| = 1 para todos os símbolos
    return symbol

# Parâmetros.
N = 10000                      # Número de símbolos
numOfClasses = 8               # São 8 classes (8-PSK)
EsN0dB = 13                    # SNR em dB
EsN0Lin = 10.0 ** (-(EsN0dB / 10.0))   # SNR linear

# Geração dos símbolos e canal AWGN (abordagem vetorizada).
# Gera os bits originais para os símbolos 8-PSK.
bit_8psk = np.random.randint(0, numOfClasses, N)

# Modula os bits em símbolos 8-PSK.
symbols = modulate8PSK(bit_8psk)

# Adiciona ruído AWGN aos símbolos.
noise = np.sqrt(EsN0Lin / 2.0) * (np.random.randn(N) + 1j * np.random.randn(N))
y = symbols + noise

# Criação da matriz de atributos (features).
X = np.c_[np.real(y), np.imag(y)]   # Features: parte real + parte imaginária

# Esplitagem em treino e teste.
X_train, X_test, y_train, y_test = train_test_split(
    X, bit_8psk, test_size=0.3, random_state=seed, stratify=bit_8psk
)

# Plotagem das constelações 8-PSK.
plt.figure(figsize=(10, 8))
colors = ['black', 'red', 'blue', 'green', 'orange', 'purple', 'brown', 'magenta']
labels = [f'Symbol {i}' for i in range(8)]

for i in range(8):
    idx = np.argwhere(bit_8psk == i)
    plt.plot(np.real(y[idx.ravel()]), np.imag(y[idx.ravel()]), '.',
             color=colors[i], label=labels[i], markersize=8)

plt.grid(True, alpha=0.3)
plt.xlabel('In-Phase (Real)', fontsize=12)
plt.ylabel('Quadrature (Imag)', fontsize=12)
plt.title(f'Constelação 8-PSK com AWGN (Es/N0 = {EsN0dB} dB)', fontsize=14)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.axis('equal')
plt.tight_layout()
plt.show()

# Informações gerais.
print(f"   Geração de dados 8-PSK concluída.")
print(f"   Total de símbolos: {N}")
print(f"   Número de classes: {numOfClasses}")
print(f"   SNR (Es/N0): {EsN0dB} dB")
print(f"   Shape de X: {X.shape}")
print(f"   Shape de y: {bit_8psk.shape}")
print(f"   Classes únicas: {np.unique(bit_8psk)}")

2. Normalmente, treina-se um modelo com uma relação sinal-ruído mediana (nem muito, nem pouco ruído, de forma que o modelo aprenda a decodificar os símbolos quando contaminados com ruído), como a usada para gerar os símbolos no item anterior.

Portanto, neste exercício, treine um **Regressor Softmax** com o conjunto de treinamento gerado no item anterior e calcula as acurácias de treinamento e validação (teste).

**DICAS:**

+ Para configurar o **Regressor Softmax**, chame seu modelo de `model` e
instancie o modelo `LogisticRegression`com os seguintes parâmetros:

```python
model = LogisticRegression(
  max_iter=1000
  random_state=seed
  n_jobs=-1
)
```

* Dado que os símbolos estão comtaminados com ruído (dados gerados com SNR mediana), seu classificador deve apresentar uma acurácia acima de 97% para ambos os conjuntos (treino e teste);

* Treine o modelo utilizando o conjunto de treinamento (`X_train`, `y_train`);

**Observação importante:**

* Nomeie o modelo treinado como `model` conforme apontado anteriormente (isso é obrigatório para os próximos exercícios funcionarem corretamente).

In [ ]:
# Digite aqui o código do exercício.

3. Calcule e plote a **Matriz de Confusão** com o conjunto de validação (teste).

In [ ]:
# Digite aqui o código do exercício.

4. Imprima as métricas calculadas pela função `classification_report` para o conjunto de validação (teste).

In [ ]:
# Digite aqui o código do exercício.

5. Usando o modelo treinado no item anterior, plote as regiões de decisão deste classificador. Plote o conjunto total (treino + validação) sobre as regiões de decisão.

In [ ]:
# Digite aqui o código do exercício.

6. Observe a figura acima com as regiões de decisão e responda:
+ Todas as amostras das 8 classes foram classificadas corretamente? Ou seja, as amostras de cada classe (símbolo 8PSK) estão em suas respectivas regiões?

**Resposta:**

<span style="color:blue">Escreva abaixo a resposta do exercício.</span>

7. Execute a célula abaixo para realizar a comparação da taxa de erro de símbolo (SER) da modulação 8-PSK com código Gray obtida pelo modelo de classificação treinado e a curva de SER teórica.

A figura gerada mostrará a:

* SER simulada (com o classificador treinado);
* SER teórica da modulação 8-PSK (a SER teórica é obtida com o detector de máxima verossimilhança, que é o detector ótimo).

Portanto, execute a célula de código abaixo e analise o resultado obtido.

**DICAS:**

+ Para que o código abaixo funcione corretamente, o nome do seu classificador treinado anteriromente deve ser `model`.

**Observação importante**

+ Como usamos simulação de monte carlo para calcular a SER do modelo de classificação, o tempo de execução da célula abaixo pode ser longo.

In [ ]:
#=========SIMULAÇÃO MONTE CARLO - SER vs Eb/N0 para 8PSK com Gray Coding========
#=====Gray Coding → menor taxa de erro de bits (BER) para a mesma SER===========
# Detector: Classificador ML treinado (model.predict)
# =============================================

# =============================================
# CONFIGURAÇÕES:
# =============================================

N = 50_000_000                     # Número de símbolos Monte Carlo

# Sweep em Eb/N0 com grid fino em alta SNR
EbN0dB = np.concatenate([
    np.arange(-2, 10, 2),
    np.arange(10, 14, 0.5),
])

k = 3                             # bits por símbolo (8-PSK)
EsN0dB = EbN0dB + 10 * np.log10(k)

ser_simu = np.zeros(len(EbN0dB))
ser_theo = np.zeros(len(EbN0dB))

# =============================================
# CONSTELAÇÃO 8PSK COM GRAY CODING:
# =============================================

# Ordem Gray para 8-PSK (padrão usado em comunicações)
gray_angles_deg = np.array([0, 45, 90, 135, 180, 225, 270, 315])

def modulate8PSK_gray(symbol_idx):
    """Modulação 8PSK com Gray Coding (energia normalizada Es = 1)"""
    theta = gray_angles_deg[symbol_idx] * np.pi / 180.0
    return np.exp(1j * theta)   # |s| = 1

# Precomputa a constelação Gray (usada apenas para geração dos símbolos)
const_gray = np.array([modulate8PSK_gray(i) for i in range(8)])

# =============================================
# FUNÇÕES AUXILIARES:
# =============================================

def Q(x):
    return 0.5 * erfc(x / np.sqrt(2))

# =============================================
# SIMULAÇÃO:
# =============================================

print("Iniciando simulação 8-PSK com Gray Coding + Detector ML (model.predict)...\n")
print(f"{'Eb/N0(dB)':>8}   {'SER Simulado':>12}   {'SER Teórico':>12}   {'Tempo (s)':>8}")
print("-" * 65)

start_total = time.time()

for idx in range(len(EbN0dB)):
    t0 = time.time()

    EsN0Lin = 10.0 ** (EsN0dB[idx] / 10.0)
    N0 = 1.0 / EsN0Lin

    # Geração de símbolos (índices 0-7 na ordem Gray)
    symbol_idx = np.random.randint(0, 8, N)
    symbol = modulate8PSK_gray(symbol_idx)

    # Canal AWGN
    noise = np.sqrt(N0 / 2.0) * (np.random.randn(N) + 1j * np.random.randn(N))
    y = symbol + noise

    # Deletando arrays que não são mais necessárias
    del symbol, noise

    # ====================== DETECTOR USANDO CLASSIFICADOR ML ======================
    # Converte símbolo recebido para formato que o modelo espera (real + imag)
    X = np.c_[np.real(y), np.imag(y)]

    # Deletando arrays que não são mais necessárias
    del y

    # Predição com o modelo treinado anteriormente (Softmax Regression, etc.)
    detected_symbol = model.predict(X)

    # SER simulada
    errors = np.sum(detected_symbol != symbol_idx)
    ser_simu[idx] = errors / N

    # SER teórica
    arg = np.sqrt(2 * EsN0Lin) * np.sin(np.pi / 8)
    ser_theo[idx] = 2 * Q(arg)

    elapsed = time.time() - t0
    print(f"{EbN0dB[idx]:8.1f}   {ser_simu[idx]:.3e}      {ser_theo[idx]:.3e}      {elapsed:7.2f}s")
print(f"\nTempo total: {time.time() - start_total:.1f} segundos")
print(f"\n Simulação concluída em {time.time() - start_total:.1f} segundos!")

# =============================================
# PLOT:
# =============================================
plt.figure(figsize=(10, 6))
plt.semilogy(EbN0dB, ser_theo, 'b-', linewidth=2.5, label='8PSK Teórica')
plt.semilogy(EbN0dB, ser_simu, 'ro', markersize=6, label='8PSK Simulada (Gray + ML Classifier)')
plt.xlabel('Eb/N0 [dB]')
plt.ylabel('Symbol Error Rate (SER)')
plt.title('8PSK com Gray Coding + Detector ML treinado\n(SER vs Eb/N0 em canal AWGN)', fontsize=14, pad=15)
plt.grid(True, which='both')
plt.legend(fontsize=12)
plt.xlim([-2, 14])
plt.ylim([9e-6, 1])
plt.show()

8. Após a análise dos resultados acima, podemos dizer que o classificador treinado apresenta uma boa performance quando comparado com a curva da taxa de erro de símbolo teórica? Ou seja, as curvas são similares? (**Justifique sua resposta**.)

**Resposta:**

<span style="color:blue">Escreva abaixo a resposta do exercício.</span>

9. A figura a seguir apresenta as regiões de decisão (também chamadas de Regiões de Voronoi) de um detector ótimo para a modulação 8-PSK, conhecido como detector de Máxima Verossimilhança (Maximum Likelihood – ML). Essas regiões são obtidas a partir de princípios clássicos de Processamento de Sinais Digitais e Teoria da Detecção, sem a necessidade de treinamento ou uso de modelos de aprendizado de máquina.

O objetivo do detector é simples e bem definido: decidir qual símbolo foi transmitido a partir do sinal recebido, minimizando a taxa de erro de símbolo (SER). Para isso, o detector ótimo realiza uma operação direta e extremamente eficiente: seleciona o ponto da constelação mais próximo do sinal recebido, com base na distância euclidiana.

Esse problema possui uma estrutura matemática completamente conhecida e bem comportada. Como consequência, o detector de Máxima Verossimilhança já é **ótimo do ponto de vista teórico**, ou seja, nenhum outro método — incluindo modelos de aprendizado de máquina — pode superá-lo em desempenho quando as condições do sistema são conhecidas.

Além disso, sua implementação é simples, direta e de **baixíssima complexidade computacional**, não exigindo fase de treinamento, ajuste de hiperparâmetros ou grandes volumes de dados.




<img style="display: block;-webkit-user-select: none;margin: auto;cursor: zoom-in;background-color: hsl(0, 0%, 90%);transition: background-color 300ms;" src="https://www.researchgate.net/profile/Michael-Fitz/publication/3158676/figure/fig1/AS:670043138428940@1536762141312/Diagram-representing-the-8PSK-signal-space.pbm" width="497" height="378">

10. Com base nas informações do item 9 acima sobre o detector de Máxima Verossimilhança (solução clássica ótima e de baixa complexidade computacional), responda:

+ Faz sentido usarmos modelos de aprendizado de máquina para resolver problemas para os quais já se têm soluções ótimas e de baixa complexidade computacional?

**DICAS:**

+ Pense sobre o ponto de vista de complexidade computacional, treinamento, necessidade de grandes volumes de dados para treinamento, otimização de hiperparâmetros, encontrar a SNR ideal para o treinamento de forma com que o modelo generalize bem, etc.

**Resposta:**

<span style="color:blue">Escreva abaixo a resposta do exercício.</span>

11. Com base na sua resposta anterior (item 10), discuta em quais cenários o uso de modelos de aprendizado de máquina se torna justificável em sistemas de comunicações digitais (por exemplo: cancelamento de interferência, sistemas MIMO massivos, equalização, beamforming adaptativo, comunicações em mmWave, entre outros). Desenvolva sua resposta de forma crítica, destacando **quando o aprendizado de máquina é realmente necessário** e quando ele pode trazer vantagens concretas em relação às abordagens clássicas.

Para orientar sua análise, considere os seguintes pontos:

* Em muitos problemas clássicos (como a detecção de símbolos em modulações conhecidas), existem soluções analíticas ótimas, simples e de baixa complexidade. Nesses casos, o uso de aprendizado de máquina pode ser desnecessário.

* Por outro lado, em cenários mais complexos — como sistemas MIMO ou Massive MIMO — o sinal recebido depende de múltiplos caminhos, interferências e de um canal altamente variável, o que torna a modelagem analítica difícil ou até inviável.

* Em aplicações como beamforming adaptativo e comunicações em ondas milimétricas (mmWave), o problema frequentemente envolve **otimizações não convexas e de alta dimensionalidade**, com restrições práticas de hardware. Nesses casos, métodos clássicos podem exigir algoritmos iterativos complexos, enquanto técnicas baseadas em aprendizado de máquina podem aproximar soluções com menor custo computacional em tempo real.

* Além disso, tarefas como estimação de canal, cancelamento de interferência e seleção de feixes envolvem ambientes dinâmicos e imprevisíveis, onde modelos baseados em dados podem capturar padrões que são difíceis de descrever analiticamente.

**DICAS:**

* Nem todas as soluções clássicas (i.e., aquelas que não utilizam aprendizado de máquina) são ótimas — muitas vezes elas são aproximações ou soluções subótimas adotadas por limitações práticas.

* O aprendizado de máquina tende a ser mais vantajoso quando:

  * o modelo matemático do sistema é desconhecido ou muito complexo;
  * o problema envolve alta dimensionalidade ou múltiplas variáveis acopladas;
  * o ambiente é dinâmico, não estacionário ou difícil de modelar;
  * há necessidade de adaptação em tempo real baseada em dados.

* Por outro lado, quando existe uma solução analítica ótima, simples e eficiente, o uso de aprendizado de máquina pode representar um aumento desnecessário de complexidade (**overkill**).

**Objetivo da questão:**
Levar você a refletir criticamente sobre **quando usar aprendizado de máquina em comunicações** — não apenas como uma ferramenta poderosa, mas como uma escolha que deve ser justificada pela natureza do problema.


**Resposta:**

<span style="color:blue">Escreva abaixo a resposta do exercício.</span>

# 3. **PROJETO DE CLASSIFICAÇÃO APLICADA À PREVISÃO DE INTENÇÃO DE COMPRA EM AMBIENTES DE E-COMMERCE.**

**CONTEXTO: Você está participando de um processo seletivo para uma empresa americana do setor de e-commerce. O objetivo dessa etapa é avaliar sua capacidade de responder perguntas a respeito do modelo de Machine Learning aplicado neste projeto.**

**O projeto em questão é capaz de prever a probabilidade de um usuário finalizar uma compra durante uma sessão de navegação no site de comércio eletrônico, com base no comportamento registrado ao longo dessa sessão.**

* **Informações sobre o Conjunto de Dados:**

Para desenvolver este projeto, você deve utilizar o dataset ***Online Shoppers Purchasing Intention Dataset***.

Este dataset contém 12.330 sessões de navegação, coletadas ao longo de um período de 1 ano, de forma que cada sessão pertença a um usuário diferente, evitando qualquer viés relacionado a campanhas específicas, dias especiais ou períodos sazonais.

Além disso, o dataset em questão é composto por 18 variáveis, sendo 10 variáveis numéricas, 6 variáveis categóricas e 2 variáveis binárias, conforme detalhado na tabela abaixo.

Uma das variáveis binárias é a receita (`revenue`), que é a variável de interesse (rótulo) que identifica se uma compra foi realizada ou não.

| Coluna                     | Tipo        | Descrição                                                                 |
|---------------------------|------------|---------------------------------------------------------------------------|
| Administrative            | Numérico   | Número de páginas administrativas visitadas                              |
| Administrative_Duration   | Numérico   | Tempo gasto em páginas administrativas                                   |
| Informational             | Numérico   | Número de páginas informativas visitadas                                 |
| Informational_Duration    | Numérico   | Tempo gasto em páginas informativas                                      |
| ProductRelated            | Numérico   | Número de páginas de produtos visitadas                                  |
| ProductRelated_Duration   | Numérico   | Tempo gasto em páginas de produtos                                       |
| BounceRates               | Numérico   | Taxa média de rejeição (usuário sai sem interagir)                       |
| ExitRates                 | Numérico   | Taxa média de saída das páginas                                          |
| PageValues                | Numérico   | Valor estimado da página para conversão                                  |
| SpecialDay                | Numérico   | Proximidade a datas especiais (ex: Black Friday, Dia dos Namorados)      |
| Month                     | Categórico | Mês da sessão                                                            |
| OperatingSystems          | Categórico | Sistema operacional do usuário                                           |
| Browser                   | Categórico | Navegador utilizado                                                     |
| Region                    | Categórico | Região geográfica do usuário                                             |
| TrafficType               | Categórico | Origem do tráfego (ex: orgânico, ads)                                   |
| VisitorType               | Categórico | Tipo de visitante (novo, recorrente, etc.)                              |
| Weekend                   | Booleano   | Indica se a sessão ocorreu no fim de semana                             |
|-------------------------|--------------- |---------------------------------------------------------------------------|
|-------------------------|--------------- | ----------------------- **A variável abaixo será o rótulo** ----------------------|
| **Revenue**                   | **Booleano**   | **Variável alvo (rótulo): indica se houve compra (True/False)**                      |



**REFERÊNCIAS:**

[1] SAKAR, C. O.; POLAT, S. O.; KATIRCIOGLU, M.; KASTRO, Y. Real-time prediction of online shoppers’ purchasing intention using multilayer perceptron and LSTM recurrent neural networks. Neural Computing and Applications, v. 31, n. 10, p. 6893–6908, 2019. DOI: 10.1007/s00521-018-3523-0.

[2] DUA, D.; GRAFF, C. UCI Machine Learning Repository: Online Shoppers Purchasing Intention Dataset. Irvine: University of California, School of Information and Computer Science, 2019. Disponível em: https://archive.ics.uci.edu/ml/datasets/Online+Shoppers+Purchasing+Intention+Dataset
. Acesso em: 12 abr. 2026.



1. Execute a célula abaixo para instalar os pacotes necessários e carregar o dataset Online Shoppers Purchasing Intention.

In [ ]:
# =============================================
# Online Shoppers Intention
# Previsão de Intenção de Compra
# =============================================
!pip install imbalanced-learn -q

import pandas as pd # Manipulação de dados
import numpy as np # Operações numéricas
import matplotlib.pyplot as plt # Gráficos básicos
import seaborn as sns # Gráficos estatísticos avançados

# Pré-processamento
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder, PolynomialFeatures

# Divisão de dados e validação
from sklearn.model_selection import train_test_split, cross_val_score, KFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay

# Modelos de classificação
from sklearn.linear_model import LogisticRegression, Perceptron

# Pré-processamento
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder, PolynomialFeatures

# Balanceamento de classes
from imblearn.over_sampling import SMOTE

# Pipelines
from sklearn.pipeline import Pipeline

# Avaliação de modelos
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    auc
)

# Outros utilitários
import pickle
import urllib

# Reseta o gerador de sequências pseudo aleatórias.
seed = 42
np.random.seed(seed)

#Baixa as bases de dados do dropbox.É uma opção usar este método
urllib.request.urlretrieve('https://www.dropbox.com/scl/fi/utviqrvn5ssr8d1xr2tyq/online_shoppers_intention.csv?rlkey=c7je5gdanksw0aw9z88uesfc7&st=tr6xibzx&dl=1', 'online_shoppers_intention.csv')

#Importa os arquivos CSV.
df = pd.read_csv('./online_shoppers_intention.csv')

# Rótulos: Converte a variável alvo para um número inteiro (0 = Falso, 1 = Verdadeiro).
df['Revenue'] = df['Revenue'].astype(int)

# Apresenta as primeiras 20 linhas do dataset.
df.head(20)

2. Execute a célula abaixo e avalie a distribuição de exemplos das duas classes mostrada na figura abaixo.

In [ ]:
df["Revenue"].value_counts().plot(kind="bar")
plt.xlabel("Classe")
plt.ylabel("Número de exemplos")
plt.title("Distribuição das classes")
plt.show()

3. Após analisar a figura acima, responda:

+ O dataset é balanceado ou desbalanceado?
+ Qual impacto isso pode ter no treinamento do modelo?

**Responda com suas palavras**

**Justifique suas respostas**

**DICAS**

+ Um conjunto de dados desbalanceado pode causar os seguintes problemas no treinamento de um modelo de aprendizado de máquina:

  + Viés para a classe majoritária: O modelo tende a prever mais a classe que aparece com maior frequência
  + Baixo desempenho na classe minoritária: Ele “aprende pouco” sobre a classe rara, gerando muitos erros (principalmente falsos negativos)
  + Métricas enganosas (accuracy paradox): Pode ter alta acurácia, mesmo errando quase todos os casos importantes
  + Dificuldade de aprendizado: Poucos exemplos da classe minoritária não são suficientes para o modelo aprender seus padrões

**Resposta:**

<span style="color:blue">Escreva abaixo a resposta do exercício.</span>



4. Dado que falsos positivos e falsos negativos têm o mesmo custo associado, qual métrica é mais adequada para este problema: acurácia ou F1-score? Por quê? (**Justifique suas respostas**)


**Resposta:**

<span style="color:blue">Escreva abaixo a resposta do exercício.</span>

5. Execute a célula abaixo para plotar um heatmap de correlação entre os atributos numéricos e a variável alvo `Revenue`.

**DICAS**

+ O heatmap é uma forma visual de ver “quem está relacionado com quem” no dataset.
+ Cada elemento mostra o quanto duas variáveis estão relacionadas.
+ Interpretação do heatmap:
  + Valores próximos de +1 → forte correlação positiva
  + Valores próximos de -1 → forte correlação negativa
  + Valores próximos de 0 → pouca relação

+ O heatmap usa cores para mostrar isso visualmente, facilitando identificar padrões

In [ ]:
# correlação de atributos numéricos apenas com a variável alvo
# Adiciona numeric_only=True para calcular a correlação apenas para colunas numéricas
corr_target = df.corr(numeric_only=True)[["Revenue"]].sort_values(by="Revenue", ascending=False)

plt.figure(figsize=(6, 8))
sns.heatmap(corr_target, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlação com a variável alvo (Revenue)")
plt.show()

6. Quais atributos apresentam maior correlação (positiva ou negativa) com `Revenue`?

**Resposta:**

<span style="color:blue">Escreva abaixo a resposta do exercício.</span>

7. Execute a célula abaixo para transformar algumas colunas categóricas em numéricas.

As colunas `Month`, `VisitorType` e `Weekend` são categóricas. O código abaixo as converte em colunas numéricas.

Os valores da coluna `Weekend` são convertidos para números inteiros 0 ou 1.

As colunas `Month` e `VisitorType`são convertidas em várias colunas binárias usando-se a codificação one-hot.

In [ ]:
# 1. One-Hot Encoding para Month e VisitorType
df = pd.get_dummies(df, columns=["Month", "VisitorType"], drop_first=True)

# 2. Converter Weekend para 0/1
df["Weekend"] = df["Weekend"].astype(int)

# visualizar o resultado da conversão
cols = [col for col in df.columns if "Month_" in col or "VisitorType_" in col or col == "Weekend"]
df[cols].head()

8. Execute a célula abaixo para pré-processar os dados:

+ Separar atributos de entrada (X) e rótulos (y)
+ Converter variáveis categóricas

In [ ]:
# variável alvo: contém apenas Revenue
y = df["Revenue"]

# entradas: contém todos os atributos EXCETO Revenue
X = df.drop("Revenue", axis=1)

# converter variáveis categóricas: Converte variáveis categóricas em números (One-Hot Encoding)
# o parâmetro "drop_first=True" remove uma coluna para evitar redundância.
X = pd.get_dummies(X, drop_first=True)

# Imprime dimensões da matriz de atributos e vetor de rótulos.
print(f'Shape de X: {X.shape}')
print(f'Shape de y: {y.shape}')

9. Divida os dados em conjuntos de **treino (80%)** e **teste (20%)** de forma estratificada.

**DICAS**
+ Estratificar é importante porque garante que treino e teste tenham a mesma proporção de classes do dataset original, evitando distorções na aprendizagem e na avaliação do modelo.
+ Para fazer a divisão estratificada com `train_test_split`, você só precisa usar o parâmetro `stratify=y`.

In [ ]:
# Digite aqui o código do exercício.

10. Padronize os atributos dos conjuntos de treinamento e de validação (teste) com `StandardScaler`.

In [ ]:
# Digite aqui o código do exercício.

11. Por que padronizamos os dados? (**Justifique sua resposta**)

**Resposta:**

<span style="color:blue">Escreva abaixo a resposta do exercício.</span>

12. Treine um regressor logístico com o conjunto de treinamento e imprima as métricas de classificação com a função `classification_report` usando o conjunto de validação (teste).

**DICAS**
+ A partir de agora, você deve usar apenas os conjuntos padronizados.

In [ ]:
# Digite aqui o código do exercício.

13. Após analisar o resultado anterior, responda:
+ O modelo está enviesado para a classe majoritária?

**DICAS**
+ Analise o recall de cada classe e verifique se o modelo consegue identificar bem ambas, considerando também a quantidade de exemplos (support).


**Resposta:**

<span style="color:blue">Escreva abaixo a resposta do exercício.</span>

14. Treine um regressor logístico com o parâmetro `class_weight='balanced'` configurado. Use o conjunto de treinamento para treinar o modelo e imprima as métricas de classificação com a função `classification_report` usando o conjunto de validação (teste).

**DICAS**

+ O parâmetro `class_weight='balanced'` é para compensar o desbalanceamento das classes durante o treinamento. Ele faz o modelo dar mais importância para a classe minoritária e menos para a majoritária.
+ O `LogisticRegression` ajusta automaticamente pesos inversamente proporcionais à frequência das classes:
  + Classe majoritária → peso menor
  + Classe minoritária → peso maior
+ Ou seja, erros na classe rara “custam mais” para o modelo


In [ ]:
# Digite aqui o código do exercício.

15. Compare os dois reportes de classificação anteriores (regressor logístico treinado com e sem o parâmetro `class_weight`) e responda:

+ Qual dos dois classificadores apresenta o melhor desempenho global?
+ Qual das duas classes apresentou melhoria em seu desempenho, fazendo com que o desempenho global melhorasse, quando o parâmetro `class_weight` foi usado?
+ Alguma das duas classes apresentou desempenho pior quando o parâmetro `class_weight` foi usado? Qual?

(**Justifique suas respostas**)

**Resposta:**

<span style="color:blue">Escreva abaixo a resposta do exercício.</span>

16. Execute a célula abaixo. Ela fará a busca por um limiar de classificação que melhore o desempenho do classificdor.

Um limiar de classificação de 0.5 só é ideal em casos de classes balanceadas. Em casos como o deste exercício, onde as classes são desbalanceadas, é provável que o limiar de classificação ótimo seja diferente de 0.5.

Portanto, neste item, faremos uma busca por um limiar de classificação que melhore o desempenho global do classificador.

Para verificar qual é o melhor limiar de classificação, o código abaixo faz:
  1. Gera probabilidades com o conjunto de validação (predict_proba)
  2. Testa vários limiares de classificação no intervalo entre 0 e 1
  3. Escolhe o limiar que maximiza o F1-score, pois é a métrica que garante equilíbrio entre falsos positivos e falsos negativos.

**DICAS**

+ Aqui, nós usamos o modelo treinado com o parâmetro `class_weight='balanced'`.
+ Para que este item execute sem problemas, o modelo de regressão logística treinado no item anterior deve-se chamar `model`.

In [ ]:
# Obtém as probabilidades previstas pelo modelo para cada amostra do conjunto de teste
y_probs = model.predict_proba(X_test_scaled)[:,1]

# Cria 1000 valores de limiar igualmente espaçados entre 0 e 1
thresholds = np.linspace(0, 1, 1000)
f1_scores = []

# Para cada possível limiar, faça...
for t in thresholds:
    # Converte probabilidades em classes (0 ou 1) usando o limiar atual
    # Se a probabilidade for maior ou igual a t → classifica como 1
    # Caso contrário → classifica como 0
    y_pred = (y_probs >= t).astype(int)
    # Calcula o F1-score entre os valores reais e os previstos
    # F1 combina precision e recall em uma única métrica
    f1_scores.append(f1_score(y_test, y_pred))

# Encontra o índice do maior F1-score obtido
best_t = thresholds[np.argmax(f1_scores)]
# Exibe o melhor limiar encontrado
print("Melhor limiar de classificação:", best_t)

# Usa melhor limiar encontrado para realizar as classificações.
y_pred_test_adjusted = (y_probs > best_t).astype(int)

# Mostra métricas detalhadas.
print('\n------- Reporte de classificação -------')
print(classification_report(y_test, y_pred_test_adjusted))

17. Após analisar os resultados acima responda

+ Qual é o melhor limiar de classificação?
+ O novo valor de limiar de classificação faz com que tenhamos menos falsos positivos ou falsos negativos?
+ O desempenho global do classificador melhorou?
+ Dado que o custo de falsos positivos e falsos negativos é igual, esse classificador é melhor do que as anteriores?

**(Justifique suas respostas**)

**Resposta:**

<span style="color:blue">Escreva abaixo a resposta do exercício.</span>